---
# **Q. Step 2 — implement K-fold CV from scratch (no sklearn)**
---


In [2]:
import numpy as np
from sklearn.datasets import load_iris        # just for real data
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

def kfold_cross_validate_scratch(X, y, model, K=5, shuffle=True, random_state=42):
    """
    Manual K-fold cross-validation.
    Returns array of validation scores, one per fold.
    """
    n = len(y)
    indices = np.arange(n)

    # Shuffle so folds aren't ordered by class or time
    if shuffle:
        rng = np.random.default_rng(random_state)
        rng.shuffle(indices)

    fold_size = n // K
    val_scores  = []
    train_scores = []

    print(f"K-fold CV (K={K}) — manual implementation")
    print(f"{'Fold':>4} | {'Train size':>10} | {'Val size':>8} | {'Train acc':>9} | {'Val acc':>8}")
    print("-" * 55)

    for fold in range(K):
        # ── 1. Slice out validation indices ──
        val_start = fold * fold_size
        # Last fold takes any remainder
        val_end   = val_start + fold_size if fold < K-1 else n
        val_idx   = indices[val_start:val_end]
        train_idx = np.concatenate([indices[:val_start], indices[val_end:]])

        # ── 2. Split ──
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # ── 3. Preprocess INSIDE the loop (critical — no leakage) ──
        scaler   = StandardScaler()
        X_train  = scaler.fit_transform(X_train)   # fit on train only
        X_val    = scaler.transform(X_val)           # transform val with train stats

        # ── 4. Train fresh model on this fold's training data ──
        model.fit(X_train, y_train)

        # ── 5. Score both sets ──
        tr_acc  = accuracy_score(y_train, model.predict(X_train))
        val_acc = accuracy_score(y_val,   model.predict(X_val))

        train_scores.append(tr_acc)
        val_scores.append(val_acc)

        print(f"  {fold+1:2d}   | {len(train_idx):>10d} | {len(val_idx):>8d} | "
              f"{tr_acc:>9.4f} | {val_acc:>8.4f}")

    print("-" * 55)
    train_scores = np.array(train_scores)
    val_scores   = np.array(val_scores)
    print(f"\nTrain — mean={train_scores.mean():.4f}  std={train_scores.std():.4f}")
    print(f"Val   — mean={val_scores.mean():.4f}  std={val_scores.std():.4f}")

    gap = train_scores.mean() - val_scores.mean()
    if gap > 0.05:
        print(f"\n⚠  Train-val gap = {gap:.4f} — possible overfitting")
    else:
        print(f"\n✓  Train-val gap = {gap:.4f} — model generalises well")

    return val_scores


# ── Run it ──
iris = load_iris()
X, y = iris.data, iris.target

model = LogisticRegression(max_iter=1000, random_state=42)
scores = kfold_cross_validate_scratch(X, y, model, K=5)

K-fold CV (K=5) — manual implementation
Fold | Train size | Val size | Train acc |  Val acc
-------------------------------------------------------
   1   |        120 |       30 |    0.9750 |   0.9333
   2   |        120 |       30 |    0.9583 |   0.9667
   3   |        120 |       30 |    0.9750 |   0.9667
   4   |        120 |       30 |    0.9750 |   0.9667
   5   |        120 |       30 |    0.9667 |   0.9667
-------------------------------------------------------

Train — mean=0.9700  std=0.0067
Val   — mean=0.9600  std=0.0133

✓  Train-val gap = 0.0100 — model generalises well


---
# **Q.  verify your scratch implementation matches sklearn exactly**
---

In [3]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

# Same experiment via sklearn — should get same mean (small diff due to how
# sklearn handles remainders and shuffling internally, but should be very close)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, random_state=42))
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
sk_scores = cross_val_score(pipe, X, y, cv=kf, scoring='accuracy')

print("sklearn cross_val_score:")
print(f"  Scores : {sk_scores.round(4)}")
print(f"  Mean   : {sk_scores.mean():.4f}")
print(f"  Std    : {sk_scores.std():.4f}")

print("\nScratch implementation:")
print(f"  Mean   : {scores.mean():.4f}")
print(f"  Std    : {scores.std():.4f}")

print(f"\nDifference in means: {abs(sk_scores.mean() - scores.mean()):.4f}  (should be < 0.02)")

sklearn cross_val_score:
  Scores : [1.     0.9667 0.9333 0.9    0.9667]
  Mean   : 0.9533
  Std    : 0.0340

Scratch implementation:
  Mean   : 0.9600
  Std    : 0.0133

Difference in means: 0.0067  (should be < 0.02)


---
# **Q. cross_val_score: the quick single-metric version**
---


In [7]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score

iris = load_iris()
X, y = iris.data, iris.target

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, random_state=42))
])
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# cross_val_score returns ONLY the test scores — shape (K,)
scores = cross_val_score(pipe, X, y, cv=kf, scoring='accuracy')

print("cross_val_score return type:", type(scores))
print("Shape:", scores.shape)
print("Fold scores:", scores.round(4))
print(f"Mean ± std : {scores.mean():.4f} ± {scores.std():.4f}")

# You can change the metric
r2_scores = cross_val_score(pipe, X, y, cv=kf, scoring='f1_weighted')
print(f"\nWith scoring='f1_weighted': {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")

# What you CANNOT do with cross_val_score:
print("\n✗ Cannot get train scores (to check overfitting)")
print("✗ Cannot get multiple metrics in one call")
print("✗ Cannot measure fit/predict time per fold")
print("→ Use cross_validate for all of the above")

cross_val_score return type: <class 'numpy.ndarray'>
Shape: (5,)
Fold scores: [1.     0.9667 0.9333 0.9    0.9667]
Mean ± std : 0.9533 ± 0.0340

With scoring='f1_weighted': 0.9533 ± 0.0339

✗ Cannot get train scores (to check overfitting)
✗ Cannot get multiple metrics in one call
✗ Cannot measure fit/predict time per fold
→ Use cross_validate for all of the above


---
# **Q. cross_validate: the full diagnostic version**
---


In [5]:
from sklearn.model_selection import cross_validate

# cross_validate returns a DICTIONARY with much more information
cv_results = cross_validate(
    pipe, X, y,
    cv=kf,
    scoring='accuracy',
    return_train_score=True,    # ← this is the key extra parameter
    return_estimator=False      # set True to get fitted models per fold
)

print("cross_validate return type:", type(cv_results))
print("Keys:", list(cv_results.keys()))
print()

# Unpack every field
for key, val in cv_results.items():
    print(f"{key:22s}: {np.round(val, 4)}")

print()
train_scores = cv_results['train_score']
test_scores  = cv_results['test_score']
fit_times    = cv_results['fit_time']

print(f"Train accuracy : {train_scores.mean():.4f} ± {train_scores.std():.4f}")
print(f"Test  accuracy : {test_scores.mean():.4f}  ± {test_scores.std():.4f}")
print(f"Overfit gap    : {train_scores.mean() - test_scores.mean():.4f}")
print(f"Avg fit time   : {fit_times.mean()*1000:.2f} ms")

cross_validate return type: <class 'dict'>
Keys: ['fit_time', 'score_time', 'test_score', 'train_score']

fit_time              : [0.0098 0.0119 0.0251 0.016  0.0114]
score_time            : [0.0021 0.0025 0.0055 0.0021 0.0021]
test_score            : [1.     0.9667 0.9333 0.9    0.9667]
train_score           : [0.9667 0.9667 0.975  0.975  0.9583]

Train accuracy : 0.9683 ± 0.0062
Test  accuracy : 0.9533  ± 0.0340
Overfit gap    : 0.0150
Avg fit time   : 14.84 ms


---
# **Q. use multiple scoring metrics in one call**
---


In [6]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier

# Binary classification — use multiple metrics at once
X_bc, y_bc = load_breast_cancer(return_X_y=True)

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  RandomForestClassifier(n_estimators=50, random_state=42))
])

kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

# Pass a LIST of scoring strings — cross_val_score cannot do this
cv_multi = cross_validate(
    pipe_rf, X_bc, y_bc,
    cv=kf5,
    scoring=['accuracy', 'f1', 'roc_auc', 'precision', 'recall'],
    return_train_score=True
)

print("Multi-metric cross_validate results:")
print(f"\n{'Metric':<20} {'Test mean':>9} {'Test std':>9} {'Train mean':>11}")
print("-" * 52)
metrics = ['accuracy', 'f1', 'roc_auc', 'precision', 'recall']
for m in metrics:
    test_key  = f'test_{m}'
    train_key = f'train_{m}'
    print(f"{m:<20} {cv_multi[test_key].mean():>9.4f} "
          f"{cv_multi[test_key].std():>9.4f} "
          f"{cv_multi[train_key].mean():>11.4f}")

print("\nNote: roc_auc is the most reliable metric here (imbalanced-ish dataset)")
print("A large train-test gap on any metric signals overfitting.")

Multi-metric cross_validate results:

Metric               Test mean  Test std  Train mean
----------------------------------------------------
accuracy                0.9578    0.0102      1.0000
f1                      0.9664    0.0082      1.0000
roc_auc                 0.9870    0.0112      1.0000
precision               0.9638    0.0144      1.0000
recall                  0.9693    0.0104      1.0000

Note: roc_auc is the most reliable metric here (imbalanced-ish dataset)
A large train-test gap on any metric signals overfitting.


#Insights:
---
>Huge train-test gap on Random Forest! Train accuracy ≈ 1.0, test ≈ 0.96. This is the overfitting signal you cannot see with cross_val_score alone. The model has essentially memorised the training folds. You would now go and tune max_depth or min_samples_leaf to reduce this gap — cross_validate just told you something important that cross_val_score hid.